# 流水优化与自动并行


## 1. 流水的收益模型

CV 融合串行执行时，一个核干活另一个核空等。流水化让上下游**错开节奏同时工作**，但收益上限由**较慢的那个核**决定：

```text
稳态节奏 = max(T_cube, T_vector)
理论加速比 ≈ (T_cube + T_vector) / max(T_cube, T_vector)
```

推论：**stage 划分越均衡（两侧耗时接近），流水填得越满**；流水建立与排空各需若干拍，迭代次数越多占比越低——小迭代次数的算子流水收益有限。


## 2. 自动 CV 并行流水：能力与用法

自动 CV 流水是 `@pl.jit` 的**编译期变换**，两个能力：把串行 CV Kernel **自动改写**为并行流水；**自动插入全部核间同步**（串行版无需手写任何 CV 同步）。

使用三步：

1. **编写 stage 函数**：把算子划分为若干计算流程，用 `@pl.pipeline.stage` 装饰。约束：stage 不能嵌套、不能有返回值；参数支持标量/定长数组/Tensor/Tile/TileGroup/元组/struct（不支持 struct_array）；
2. **声明跨核共享 Buffer**：两类——核私有 UB/L1 用 `make_tile_group`（配 `fwd_ids`/`bwd_ids`）；共享 GM 用 `make_tensor` 声明 workspace（按 depth 等分为 slot）。`fwd_ids`=正向同步（生产者写完通知消费者）、`bwd_ids`=反向（消费者用完通知覆写）——**只配 fwd 不配 bwd 时，不保证消费者读完才覆写**。声明必须在 kernel 体内、stage 外；
3. **主循环调用 stage，并显式启用 `pipeline=pl.pipeline.PipelineConfig(preload=0)`**，先验证串行同步，再逐步增加 preload；仅加 stage 装饰器不会启用流水变换。

关键约束（节选）：

- 同一流水的 stage 必须位于同一个正步长循环，各 section 只有一次 stage 调用、严格 CV 交替；首尾 stage 间不插其他语句，不支持随运行期值变化的 stage 分支。
- 一个跨核 Buffer 恰好被**相邻两个 stage**使用（一对一生产/消费）；更远传递逐级交接；
- fwd/bwd_ids ∈ [0,15]，数量等于 slot 数或为 1（共用会串行化交接、可能降性能）；
- **slot 数下限**：写它的 stage 编号为奇数至少 2 个、偶数至少 preload 个，不足则编译期报错；
- 自动流水不扫描 `make_tile` 声明的 Tile（不插同步也不查地址）。


## 3. 与手动流水的关系

第 5 章 05.03 的手动流水是理解机制的正确路径；自动流水是满足约束时的效率选择。**先手动会排、再自动提效**；自动流水产物 `pipeline_generated.py` 可在 build 目录检查 stage 排布与同步插入。


### 把第 5 章例子实际拆成两个 stage

参考实现的 stage 1 完成输入搬运与 GEMM，用 `DualModeSplitM` 将 L0C 结果分别交给两个 AIV；stage 2 各处理 64 行，完成 Bias、GELU 与输出。

交接使用两槽 UB TileGroup，每槽 [64,128] FP32，每个 AIV 共预留 64 KiB，配置 mutex_ids=[8,9]、fwd_ids=[0,1]、bwd_ids=[2,3]。两个 stage 各调用一次 next()，由自动流水建立跨核同步。该实现不需要 GM workspace。

GM workspace 自动流水需要安装包支持 make_tensor 的 depth/fwd_ids/bwd_ids 参数，不能仅凭版本号推断支持。本例使用片上交接；与第 5 章 GM 基线比较时，同时改变了交接路径，不能把差异全部归因于自动流水调度。若要单独比较调度收益，应先统一数据通路。

配置 PipelineConfig(preload=1)，检查生成的 pipeline_generated.py 中槽位轮转、stage 次数与首尾等待，并进行数值回归。


## 4. 课后练习

1. （填空）流水稳态节奏由 ______ 决定；两侧耗时 1:3 时理论加速比约 ______。
2. （选择）只配 fwd_ids 不配 bwd_ids 的风险：
   A. 编译报错<br/>B. 消费者可能读到没写完的数据<br/>C. 生产者可能覆写未读完的数据
3. （实践）把第 5 章实践三的手动两槽流水（GEMM+Bias+GELU）对照本节约束评估：能否改用自动流水？逐条核对并给出结论。

In [ ]:
!cat ./answer/08.05_answer.txt

## 5. 本节小结

- 收益模型：稳态 = max(两侧)，均衡才满载；
- 自动流水 = stage 函数 + fwd/bwd_ids 跨核 Buffer + 编译期变换（排布 + 同步全自动）；
- 约束速记：相邻交接、ids∈[0,15]、slot 下限、make_tile 不扫描；
- 手动会排、自动提效、产物可查。